## Communication Topology and Belief Dynamics in Multi-Agent LLM Reasoning
### Experiment Analysis & Visualisation

This notebook analyses experimental results from a multi-agent LLM system tested across different communication topologies on the GSM8K benchmark.

**Topologies tested:**
- **Independent**: agents reason alone, answers aggregated via majority vote
- **Fully Connected**: all agents see each other's responses before revising
- **Mediator**: a mediator summarises responses; agents see only the summary
- **Chain**: agents answer sequentially, each seeing only the previous agent

**Primary questions:**
1. Does collaboration improve accuracy over independent reasoning?
2. How do different communication structures affect convergence?
3. What are the cost/accuracy trade-offs across topologies?

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import pandas as pd
import numpy as np
import warnings
import glob
from pathlib import Path
warnings.filterwarnings("ignore")

---
### 1. Setup & Data Loading

In [2]:
pio.templates.default = "plotly_white"

TOPO_COLORS = {
    "independent": "#636EFA",
    "full": "#EF553B",
    "mediator": "#00CC96",
    "chain": "#AB63FA",
}

AGENT_COLORS = {
    "gemma3:4b": "#FF6B6B",
    "phi4-mini": "#4ECDC4",
    "llama3.2:3b": "#45D15A",
    "qwen2.5:3b-instruct": "#F7DC6F",
}

AGENT_NAME_MAP = {
    0: "gemma3:4b", 
    1: "phi4-mini", 
    2: "llama3.2:3b", 
    3: "qwen2.5:3b-instruct"
}

In [3]:
# Load and concatenate all results into a single DataFrame
# 4 Agents, 3 Rounds, 150 Questions, Seed 0
folder = "A4-R5-Q100-S0"
result_files = list(Path(f"../results/{folder}").glob("*.csv"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f}")
df = pd.concat((pd.read_csv(f) for f in result_files), ignore_index=True)

Found 4 result files:
  ..\results\A4-R5-Q100-S0\chain_20260314_045634.csv
  ..\results\A4-R5-Q100-S0\full_20260315_194735.csv
  ..\results\A4-R5-Q100-S0\independent_20260313_213131.csv
  ..\results\A4-R5-Q100-S0\mediator_20260315_150046.csv


In [4]:
# Data overview
print("===== Dataset Summary =====")
print(f"Shape: {df.shape}")
print("\nTopology counts:")
print(df['topology'].value_counts())
print("\nRounds per topology:")
print(df.groupby('topology')['round'].max())
print("\nTemperature:        0.4 \nSamples questions:  200")
print(f"\nParse failure rate: {df['parse_failed'].mean():.2%}")

parse_by_model = df.groupby(["topology", "model"])["parse_failed"].mean().unstack()
print(parse_by_model.applymap(lambda x: f"{x:.1%}"))

===== Dataset Summary =====
Shape: (6000, 15)

Topology counts:
topology
full           2000
mediator       2000
chain          1600
independent     400
Name: count, dtype: int64

Rounds per topology:
topology
chain          4
full           5
independent    1
mediator       5
Name: round, dtype: int64

Temperature:        0.4 
Samples questions:  200

Parse failure rate: 8.00%
model       gemma3:4b llama3.2:3b phi4-mini qwen2.5:3b-instruct
topology                                                       
chain            3.0%       26.8%      1.2%                0.0%
full             2.4%       26.8%      0.6%                0.0%
independent      5.0%       41.0%      1.0%                0.0%
mediator         2.4%       29.2%      0.4%                0.0%


---
### 2. Accuracy Comparison Across Topologies

The fundamental question: **does collaboration help?**
We compare group-level accuracy (majority vote) across all topologies.

In [5]:
# Question-level accuracy table (one row per question per topology)
q_level = (
    df.groupby(["topology", "question_idx"])
    .agg(
        correct=("correct", "first"),
        expected=("expected_answer", "first"),
        group_answer=("group_answer", "first"),
        last_round_idx=("round", "idxmax"),
    )
    .reset_index()
)

last_round = df.loc[q_level["last_round_idx"]]

In [6]:
accuracy = q_level.groupby("topology")["correct"].mean().reset_index()
accuracy.columns = ["topology", "accuracy"]

In [7]:
fig = px.bar(
    accuracy,
    y="topology", x="accuracy",
    color="topology", color_discrete_map=TOPO_COLORS,
    text=accuracy["accuracy"].map("{:.1%}".format),
    title="Group Accuracy by Communication Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology"},
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_range=[0, 1])

fig.show()
fig.write_image("figures/accuracy_by_topology.png", width=1500, scale=3)

#### Individual Agent Accuracy vs Group Accuracy

Does (majority) voting actually help? Comparing individual agent accuracy to the group's majority-vote accuracy.

In [8]:
# All rows with max round per group (one per agent)
last_round = df.loc[df["round"] == df.groupby(["topology", "question_idx"])["round"].transform("max")]

# Individual agent accuracy
individual_acc = (
    last_round.assign(correct=lambda d: d["answer"] == d["expected_answer"])
    .groupby(["topology", "agent_id"])["correct"].mean()
    .reset_index(name="accuracy")
)
individual_acc["agent_id"] = individual_acc["agent_id"].map(AGENT_NAME_MAP)

# Group accuracy
group_acc = (
    q_level.groupby("topology")["correct"].mean()
    .reset_index(name="accuracy")
    .assign(agent_id="Group Vote")
)

# Combining individual and group accuracy
combined = pd.concat([individual_acc, group_acc], ignore_index=True)
agent_color_map = {**AGENT_COLORS, "Group Vote": "#000000"}


fig = px.bar(
    combined, x="topology", y="accuracy",
    color="agent_id", barmode="group",
    title="Individual Agent vs Group Accuracy by Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology", "agent_id": ""},
    text=combined["accuracy"].map("{:.1%}".format),
    category_orders={"topology": ["independent", "full", "mediator", "chain"]},
    color_discrete_map=agent_color_map
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 1.1])

fig.show()
fig.write_image("figures/individual_vs_group_accuracy.png", scale=3, width=1500)

Collaboration dramatically boosts all models, every agent jumps from 11-26% (independent) to 45-83% under collaborative topologies. However, majority voting only outperforms the best individual agent in mediator (84% vs 83%). In fully connected, chain, and independent, at least one or two agents individually beat the group vote, suggesting that open discussion and sequential passing can dilute strong individual reasoning. Mediator's structured summary appears to be the only topology where aggregation consistently adds value beyond the best single agent.

---
### 3. Confidence Analysis

Are agents overconfident? Does confidence actually predict correctness?

In [9]:
calibration = (
    last_round.groupby("topology")
    .agg(confidence=("confidence", "mean"), accuracy=("correct", lambda x: x.mean() * 100))
    .reset_index()
    .melt(id_vars="topology", var_name="Metric", value_name="value")
)

In [10]:
# Calibration Gap

# If the confidence bar is much taller than the accuracy bar, agents are **overconfident**
# A well-calibrated agent would have these roughly equal

calibration = (
    last_round.groupby("topology")
    .apply(lambda g: pd.Series({
        "Mean Confidence": g["confidence"].mean(),
        "Actual Accuracy (%)": (g["answer"] == g["expected_answer"]).mean() * 100,
    }))
    .reset_index()
)
calibration_melted = calibration.melt(
    id_vars="topology", var_name="Metric", value_name="value"
)

fig = px.bar(
    calibration_melted,
    y="topology", x="value",
    color="Metric", barmode="group",
    text=calibration_melted["value"].apply(lambda x: f"{x:.1f}%"),
    title="Calibration Gap: Mean Confidence vs Actual Accuracy",
    color_discrete_map={"Actual Accuracy (%)": "#2ecc71", "Mean Confidence": "#e74c3c"},
    labels={"value": "Percentage", "topology": "Topology"},
)
fig.update_traces(textposition="outside")

fig.show()
fig.write_image("figures/calibration_gap.png", width=1200, scale=3)

Agents claimed 90%+ confidence across all topologies regardless of correctness. 
* Confidence is **not a reliable signal** in these small models.

---
### 4. Convergence Dynamics
For multi-round topologies (fully connected, mediator), do agents converge
toward agreement? And does that agreement move toward the **correct** answer?

In [11]:

multi_round = df[df["topology"].isin(["full", "mediator"])]

# Agreement per round
avg_agreement = (
    multi_round.groupby(["topology", "question_idx", "round"])["answer"]
    .apply(lambda a: (a.dropna() == a.dropna().mode().iloc[0]).mean() if len(a.dropna()) else 0)
    .reset_index(name="agreement")
    .groupby(["topology", "round"], as_index=False)["agreement"].mean()
)

# Group accuracy per round
avg_acc_round = (
    multi_round.groupby(["topology", "question_idx", "round"])
    .apply(lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0)
    .reset_index(name="correct")
    .groupby(["topology", "round"], as_index=False)["correct"].mean()
)

conf_round = (multi_round.groupby(["topology", "round"])["confidence"].mean().reset_index())


# Subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Agent Agreement Rate Across Rounds", "Group Accuracy Across Rounds", "Mean Confidence Across Rounds")
)

for topo in avg_agreement["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_agreement[avg_agreement["topology"] == topo]["round"],
            y=avg_agreement[avg_agreement["topology"] == topo]["agreement"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo))
        ),
        row=1, col=1
    )

for topo in avg_acc_round["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_acc_round[avg_acc_round["topology"] == topo]["round"],
            y=avg_acc_round[avg_acc_round["topology"] == topo]["correct"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo)),
            showlegend=False
        ),
        row=1, col=2
    )

for topo in conf_round["topology"].unique():
    d = conf_round[conf_round["topology"] == topo]
    fig.add_trace(
        go.Scatter(
            x=d["round"], y=d["confidence"],
            name=topo, line=dict(color=TOPO_COLORS[topo]),
            showlegend=False
        ),
        row=1, col=3
    )

fig.update_layout(
    yaxis_tickformat=".0%", yaxis_range=[0, 1.05],
    yaxis2_tickformat=".0%", yaxis2_range=[0, 1.05],
    yaxis3_range=[0, 100]
)

fig.show()
fig.write_image("figures/agreement_accuracy_across_rounds.png", width=1700, scale=3)

Both topologies show strong convergence. Agreement rises from ~45% in round 1 to ~78% by round 3, with accuracy following the same upward trend (21% -> 80%). Confirming agents are converging toward correct answers, not just blindly agreeing. Notably, mediator reaches higher accuracy faster in round 2 and 3, suggesting the mediator's structured summary helps agents focus on the right answer more efficiently. Meanwhile, mean confidence barely changes (~93% throughout), reinforcing that confidence, in the way I've implemented it, is not a meaningful signal in these models.

---
### 5. Answer Flow Between Rounds (Sankey)

How do agents change between rounds? Tracks transitions: correct->correct, correct->incorrect, incorrect->correct, incorrect->incorrect.

In [12]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "sankey"}, {"type": "sankey"}]],
    subplot_titles=["Full", "Mediator"],
)

for i, topo in enumerate(["full", "mediator"]):
    topo_df = multi_round[multi_round["topology"] == topo]
    
    transitions = {"C->C": 0, "C->I": 0, "I->C": 0, "I->I": 0}
    for r in range(1, topo_df["round"].max()):
        curr = topo_df[topo_df["round"] == r].set_index(["question_idx", "agent_id"])
        nxt = topo_df[topo_df["round"] == r + 1].set_index(["question_idx", "agent_id"])
        joined = curr[["answer", "expected_answer"]].join(nxt[["answer"]], rsuffix="_next", how="inner")
        
        c = joined["answer"] == joined["expected_answer"]
        n = joined["answer_next"] == joined["expected_answer"]
        transitions["C->C"] += (c & n).sum()
        transitions["C->I"] += (c & ~n).sum()
        transitions["I->C"] += (~c & n).sum()
        transitions["I->I"] += (~c & ~n).sum()

    values = [transitions["C->C"], transitions["C->I"], transitions["I->C"], transitions["I->I"]]
    total = sum(values)
    pct = [v / total * 100 for v in values]
    
    fig.add_trace(go.Sankey(
        node=dict(
            pad=20, thickness=25,
            label=[
                f"Correct (R1)\n{(transitions['C->C'] + transitions['C->I']) / total:.0%}",
                f"Incorrect (R1)\n{(transitions['I->C'] + transitions['I->I']) / total:.0%}",
                f"Correct (R3)\n{(transitions['C->C'] + transitions['I->C']) / total:.0%}",
                f"Incorrect (R3)\n{(transitions['C->I'] + transitions['I->I']) / total:.0%}",
            ],
            color=["#2ecc71", "#e74c3c", "#2ecc71", "#e74c3c"],
        ),
        link=dict(
            source=[0, 0, 1, 1], target=[2, 3, 2, 3],
            value=values,
            label=[f"{v:,}" for v in values],
            color=["rgba(46,204,113,0.4)", "rgba(231,76,60,0.4)",
                   "rgba(46,204,113,0.4)", "rgba(231,76,60,0.4)"],
        ),
    ), row=1, col=i + 1)

fig.update_layout(title="Answer Transitions Between Rounds")

fig.show()
fig.write_image("figures/sankey_transitions.png", width=1700, scale=3)

Both topologies show a net positive correction effect. Starting from ~36-37% individual correctness in round 1, agents improve to ~60-62% by round 3, a ~25 percentage point gain through discussion alone. The dominant flow is Incorrect->Correct, confirming that collaboration helps agents fix mistakes. A small Correct->Incorrect flow exists in both (agents occasionally getting talked out of right answers), but it is far outweighed by the corrections. The two topologies perform similarly here, though mediator shows a slightly thinner C->I band, suggesting the structured summary helps correct agents from being swayed. 
* Note: that final group accuracy (75-85%) exceeds the individual 60-62% shown here because majority voting further filters out remaining incorrect minority answers.

---
### 6. Per-Question Error Analysis

Which questions were hardest? Did certain topologies rescue questions that others got wrong?

**How to read:** 
* Diagonal = total questions solved. 
* Off-diagonal = questions solved by row but NOT column.

In [13]:

q_pivot = q_level.pivot_table(index="question_idx", columns="topology", values="correct", aggfunc="first").fillna(0)

topos = [t for t in ["independent", "full", "mediator", "chain"] if t in q_pivot.columns]
solved_by = {t: set(q_pivot[q_pivot[t] == 1].index) for t in topos}

overlap_data = []
for t1 in topos:
    for t2 in topos:
        if t1 == t2:
            overlap_data.append({"from": t1, "to": t2, "value": len(solved_by[t1])})
        else:
            overlap_data.append({"from": t1, "to": t2, "value": len(solved_by[t1] - solved_by[t2])})

overlap_df = pd.DataFrame(overlap_data).pivot(index="from", columns="to", values="value")
overlap_df = overlap_df.loc[topos, topos]

fig = px.imshow(
    overlap_df,
    text_auto=True,
    color_continuous_scale="RdYlGn",
    title="Question Overlap Between Topologies",
    labels=dict(x="Topology", y="Topology", color="Questions"),
)

fig.show()
fig.write_image("figures/question_overlap.png", scale=3)

Mediator is the dominant topology. It solves the most questions (127) and is a complete superset of independent, every question independent solved, mediator also solved (off-diagonal = 0). Full comes second with 113 but only 12 of those are unique to it; mediator captures almost everything full does. Chain solves 71 but only 2 are unique compared to mediator. All suggesting that structured mediation provides the broadest coverage, and combining topologies would offer minimal additional gain over mediator alone.

In [14]:
token_usage = (
    df.groupby(["topology", "question_idx"])
    .agg(
        total_prompt=("prompt_tokens", "sum"),
        total_completion=("completion_tokens", "sum"),
        correct=("correct", "first"),
    )
    .assign(total_tokens=lambda d: d["total_prompt"] + d["total_completion"])
    .reset_index()
)

In [15]:
# Per round accuracy and cumulative tokens for iterative topologies
scatter_rows = []
for topo in ["full", "mediator"]:
    topo_df = df[df["topology"] == topo]
    for r in range(1, topo_df["round"].max() + 1):
        up_to_r = topo_df[topo_df["round"] <= r]
        round_r = topo_df[topo_df["round"] == r]
        tokens = up_to_r.groupby("question_idx")[["prompt_tokens", "completion_tokens"]].sum().sum(axis=1).mean()
        acc = (
            round_r.groupby("question_idx")
            .apply(lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0)
            .mean()
        )
        scatter_rows.append({"topology": topo, "round": r, "label": f"{topo} R{r}", "accuracy": acc, "tokens": tokens})

# Single points for non iterative topologies
for topo in ["independent", "chain"]:
    topo_t = token_usage[token_usage["topology"] == topo]
    scatter_rows.append({"topology": topo, "round": 1, "label": topo, "accuracy": topo_t["correct"].mean(), "tokens": topo_t["total_tokens"].mean()})

scatter_df = pd.DataFrame(scatter_rows)

In [16]:

fig = px.scatter(
    scatter_df, x="tokens", y="accuracy",
    color="topology", color_discrete_map=TOPO_COLORS, text="label",
    title="Accuracy vs Token Cost (Per Round for Iterative Topologies)",
    labels={"tokens": "Mean Tokens per Question", "accuracy": "Accuracy"},
)
fig.update_traces(textposition="top center")
fig.update_layout(yaxis_tickformat=".0%", showlegend=False)

fig.show()
fig.write_image("figures/accuracy_vs_cost.png", width=1200, scale=3)

---
### 8. Summary Table

In [17]:
rows = []
for topo in ["independent", "chain"]:
    row = scatter_df[scatter_df["topology"] == topo].iloc[0]
    accuracy = row['accuracy']
    tokens = row['tokens']
    rows.append({
        "Topology": topo.title(),
        "Accuracy": f"{accuracy:.1%}",
        "Tokens/Q": f"{tokens:,.0f}",
        "Tokens/Correct": f"{tokens / accuracy:,.0f}",
        "Parse Fail %": f"{df[df['topology'] == topo]['parse_failed'].mean():.1%}"
    })

for topo in ["mediator", "full"]:
    topo_rows = scatter_df[scatter_df["topology"] == topo]
    pf = f"{df[df['topology'] == topo]['parse_failed'].mean():.1%}"
    for r in [2, 3, topo_rows["round"].max()]:
        row = topo_rows[topo_rows["round"] == r].iloc[0]
        accuracy = row['accuracy']
        tokens = row['tokens']
        rows.append({
            "Topology": f"{topo.title()}: round {r}",
            "Accuracy": f"{accuracy:.1%}",
            "Tokens/Q": f"{tokens:,.0f}",
            "Tokens/Correct": f"{tokens / accuracy:,.0f}",
            "Parse Fail %": pf
        })

summary = pd.DataFrame(rows)

In [18]:
fig = go.Figure(go.Table(
    header=dict(values=list(summary.columns), fill_color="#2c3e50",
                font=dict(color="white", size=14), align="center"),
    cells=dict(values=[summary[col] for col in summary.columns],
               font=dict(size=13), align="center", height=30),
))

fig.update_layout(title="Experiment Summary", height=500, margin=dict(b=0))

fig.show()
fig.write_image("figures/experiment_summary_table.png", width=1000, scale=3)

Mediator R2 is the clear sweet spot, 78% accuracy at just 12k tokens/Q and the lowest cost per correct answer (15,527). Adding more rounds doubles the token cost with diminishing returns: R3 gains 5% accuracy for 4k more tokens, while R5 actually drops 1% accuracy at double the cost of R2. Fully connected follows the same pattern but consistently lags behind mediator at every round. Independent is deceptively cheap (4k tokens) but its 21k tokens/correct is higher than mediator R2 because it only gets 19% right, chain is the worst value, highest tokens/correct (35k) at mediocre accuracy (61%).